In [5]:
# ==========================================
# STEP 1: Import Libraries
# ==========================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# ==========================================
# STEP 2: Load Dataset (Using Uploaded Path)
# ==========================================

df = pd.read_excel(r"C:\Users\aishh\Downloads\Excel Q1.xlsx")

# ==========================================
# STEP 3: Check Column Names
# ==========================================

print("Columns in Dataset:")
print(df.columns)

# ==========================================
# STEP 4: Automatically Detect Date Column
# ==========================================

date_column = None
for col in df.columns:
    if 'time' in col.lower() or 'date' in col.lower():
        date_column = col
        break

print("Detected Date Column:", date_column)

# Convert to datetime
df[date_column] = pd.to_datetime(df[date_column])
df['Day'] = df[date_column].dt.date

# ==========================================
# STEP 5: Create Daily Total Production per Plant
# ==========================================

daily_data = df.groupby(['Day', 'Plant'])['ProductionUnits'].sum().reset_index()

# ==========================================
# STEP 6: Define Manual MAPE Function
# ==========================================

def calculate_mape(actual, forecast):
    actual, forecast = np.array(actual), np.array(forecast)
    return np.mean(np.abs((actual - forecast) / actual)) * 100

# ==========================================
# STEP 7: Forecasting Per Plant
# ==========================================

plants = daily_data['Plant'].unique()
results = {}

for plant in plants:
    
    plant_data = daily_data[daily_data['Plant'] == plant].copy()
    plant_data = plant_data.sort_values('Day')
    
    # Split last 14 days as test
    train = plant_data.iloc[:-14].copy()
    test = plant_data.iloc[-14:].copy()
    
    # ---------------- BASELINE MODEL ----------------
    last_7_avg = train['ProductionUnits'].tail(7).mean()
    baseline_forecast = [last_7_avg] * 14
    
    baseline_mape = calculate_mape(
        test['ProductionUnits'], baseline_forecast)
    
    # ---------------- ML MODEL (Linear Regression) ----------------
    train['t'] = np.arange(len(train))
    test['t'] = np.arange(len(train), len(train) + 14)
    
    X_train = train[['t']]
    y_train = train['ProductionUnits']
    
    X_test = test[['t']]
    y_test = test['ProductionUnits']
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    ml_forecast = model.predict(X_test)
    
    ml_mape = calculate_mape(y_test, ml_forecast)
    
    results[plant] = {
        'Baseline MAPE (%)': baseline_mape,
        'ML Model MAPE (%)': ml_mape
    }

# ==========================================
# STEP 8: Print MAPE Comparison
# ==========================================

print("\n===== MAPE Comparison =====")

for plant in results:
    print(f"\nPlant: {plant}")
    print("Baseline MAPE:", round(results[plant]['Baseline MAPE (%)'], 2), "%")
    print("ML Model MAPE:", round(results[plant]['ML Model MAPE (%)'], 2), "%")

# ==========================================
# STEP 9: Final 14-Day Forecast Using ML Model
# ==========================================

print("\n===== Next 14 Days Forecast =====")

for plant in plants:
    
    plant_data = daily_data[daily_data['Plant'] == plant].copy()
    plant_data = plant_data.sort_values('Day')
    
    plant_data['t'] = np.arange(len(plant_data))
    
    X = plant_data[['t']]
    y = plant_data['ProductionUnits']
    
    model = LinearRegression()
    model.fit(X, y)
    
    future_t = np.arange(len(plant_data), len(plant_data) + 14).reshape(-1,1)
    
    future_forecast = model.predict(future_t)
    
    print(f"\nPlant: {plant}")
    print(np.round(future_forecast, 2))

Columns in Dataset:
Index(['Timestamp', 'MachineID', 'Plant', 'Temperature', 'Vibration',
       'Pressure', 'EnergyConsumption', 'ProductionUnits', 'DefectCount',
       'MaintenanceFlag', 'Hour', 'Date ', 'Utilization'],
      dtype='object')
Detected Date Column: Timestamp

===== MAPE Comparison =====

Plant: Plant_A
Baseline MAPE: 146.12 %
ML Model MAPE: 124.09 %

Plant: Plant_B
Baseline MAPE: 62.31 %
ML Model MAPE: 73.56 %

Plant: Plant_C
Baseline MAPE: 31.42 %
ML Model MAPE: 36.65 %

===== Next 14 Days Forecast =====

Plant: Plant_A
[950.4  950.33 950.26 950.19 950.12 950.05 949.99 949.92 949.85 949.78
 949.71 949.64 949.58 949.51]

Plant: Plant_B
[1010.73 1010.77 1010.8  1010.83 1010.86 1010.89 1010.93 1010.96 1010.99
 1011.02 1011.05 1011.09 1011.12 1011.15]

Plant: Plant_C
[983.77 983.71 983.65 983.59 983.54 983.48 983.42 983.36 983.3  983.24
 983.19 983.13 983.07 983.01]
